<a href="https://colab.research.google.com/github/machancejoy-max/colab-git-demo-JOY/blob/main/Final_project_PAAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Download the Sentiment140 ZIP file
!wget https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip

# Unzip the dataset
!unzip trainingandtestdata.zip


--2026-05-13 13:19:50--  https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 81363704 (78M) [application/zip]
Saving to: ‘trainingandtestdata.zip’

trainingandtestdata 100%[===================>]  77.59M  18.0MB/s    in 4.1s    

2026-05-13 13:19:55 (19.1 MB/s) - ‘trainingandtestdata.zip’ saved [81363704/81363704]

Archive:  trainingandtestdata.zip
  inflating: testdata.manual.2009.06.14.csv  
  inflating: training.1600000.processed.noemoticon.csv  


In [2]:
import pandas as pd

cols = ["target", "id", "date", "flag", "user", "text"]

df = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    names=cols
)

df.head()


,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [3]:
import re

def clean_tweet(text):
    text = text.lower()  # lowercase
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)  # remove URLs
    text = re.sub(r"@\w+", "", text)  # remove mentions
    text = re.sub(r"#\w+", "", text)  # remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove special characters
    text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces
    return text

df["clean_text"] = df["text"].apply(clean_tweet)

df[["text", "clean_text"]].head()


,text,clean_text
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",a thats a bummer you shoulda got david carr of...
1,is upset that he can't update his Facebook by ...,is upset that he cant update his facebook by t...
2,@Kenichan I dived many times for the ball. Man...,i dived many times for the ball managed to sav...
3,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire
4,"@nationwideclass no, it's not behaving at all....",no its not behaving at all im mad why am i her...


In [4]:
df["sentiment"] = df["target"].map({
    0: "negative",
    2: "neutral",
    4: "positive"
})

df["sentiment"].value_counts()


,count
sentiment,
negative,800000
positive,800000


In [5]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english", max_features=5000)
X = vectorizer.fit_transform(df["clean_text"])

X.shape


(1600000, 5000)

In [6]:
from sklearn.model_selection import train_test_split

sample_df = df.sample(200000, random_state=42)  # 200k tweets

X = sample_df["clean_text"]
y = sample_df["sentiment"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english", max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)


In [8]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)
model.fit(X_train_vec, y_train)


LogisticRegression(max_iter=200)

In [9]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_pred = model.predict(X_val_vec)

acc = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred, average="weighted")

print("Accuracy:", acc)
print("F1-Score:", f1)
print("\nClassification Report:\n", classification_report(y_val, y_pred))


Accuracy: 0.758175
F1-Score: 0.7579137720756298

Classification Report:
               precision    recall  f1-score   support

    negative       0.78      0.72      0.75     20088
    positive       0.74      0.79      0.77     19912

    accuracy                           0.76     40000
   macro avg       0.76      0.76      0.76     40000
weighted avg       0.76      0.76      0.76     40000



In [16]:
%%writefile app.py
from flask import Flask, request, jsonify
import joblib
import re

# Load model + vectorizer
model = joblib.load("logistic_model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

app = Flask(__name__)

def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    tweet = data.get("text", "")

    cleaned = clean_tweet(tweet)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]

    return jsonify({"sentiment": pred})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)



Overwriting app.py


In [15]:
import joblib

joblib.dump(model, "logistic_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")


['vectorizer.pkl']

In [17]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY logistic_model.pkl .
COPY vectorizer.pkl .

EXPOSE 5000

CMD ["python", "app.py"]


Writing Dockerfile


In [18]:
%%writefile requirements.txt
flask
scikit-learn
joblib
numpy
pandas


Writing requirements.txt
